In [ ]:
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

# Setup device and dtype
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load model and processor
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-large",
    torch_dtype=torch_dtype,
    trust_remote_code=True,
    attn_implementation="sdpa"
).to(device)

processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-large",
    trust_remote_code=True
)

def get_bbox_proposals(image):
    """
    Get class-agnostic bounding boxes from an image.
    
    Args:
        image (PIL.Image): Input image
        
    Returns:
        list: List of bboxes in format [[x1, y1, x2, y2], ...]
    """
    prompt = "<REGION_PROPOSAL>"
    
    # Process inputs
    inputs = processor(
        text=prompt, 
        images=image, 
        return_tensors="pt"
    ).to(device, torch_dtype)
    
    # Generate predictions
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
        do_sample=False
    )
    
    # Decode and post-process
    generated_text = processor.batch_decode(
        generated_ids, 
        skip_special_tokens=False
    )[0]
    
    # Parse output to get bboxes
    parsed_answer = processor.post_process_generation(
        generated_text,
        task="<REGION_PROPOSAL>",
        image_size=(image.width, image.height)
    )
    
    # Extract bbox list
    bboxes = parsed_answer["<REGION_PROPOSAL>"]["bboxes"]
    return bboxes

/cluster/scratch/niacobone/sam3/myenv/lib/python3.12/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ImportError: This modeling file requires the following packages that were not found in your environment: flash_attn. Run `pip install flash_attn`

In [ ]:
# Load image
image = Image.open("/cluster/scratch/niacobone/sam3/assets/videos/0001/0.jpg")

# Get bboxes
bboxes = get_bbox_proposals(image)
print(f"Found {len(bboxes)} objects")
print(f"Bboxes: {bboxes}")